In [ ]:
# ============================================================
# General Solution of a Second-Order Difference Equation
#
# y[n] - (3/4)y[n-1] + (1/8)y[n-2]
#     = (5/8)sin(pi*n/2)u[n]
#
# Initial conditions:
# y[-1] = 1, y[-2] = 1
# ============================================================

import sympy as sp
from IPython.display import display, Markdown

# ============================================================
# 1. Symbolic variables
# ============================================================

n = sp.symbols('n', integer=True)
lam = sp.symbols('lambda')
C1, C2, C3, C4 = sp.symbols('C1 C2 C3 C4')

# ============================================================
# 2. Homogeneous solution
# ============================================================

display(Markdown("## 1. Homogeneous equation"))

characteristic_equation = sp.Eq(lam**2 - sp.Rational(3,4)*lam + sp.Rational(1,8), 0)
display(characteristic_equation)

roots = sp.solve(characteristic_equation, lam)
lambda1, lambda2 = roots

display(sp.Eq(sp.Symbol('lambda_1'), lambda1))
display(sp.Eq(sp.Symbol('lambda_2'), lambda2))

y_h = C1*lambda1**n + C2*lambda2**n
display(sp.Eq(sp.Symbol('y_h[n]'), y_h))

# ============================================================
# 3. Particular solution
#
# Since the input is sinusoidal, we assume:
#
# y_p[n] = C3*cos(pi*n/2) + C4*sin(pi*n/2)
# ============================================================

display(Markdown("## 2. Particular solution"))

y_p = C3*sp.cos(sp.pi*n/2) + C4*sp.sin(sp.pi*n/2)
display(sp.Eq(sp.Symbol('y_p[n]'), y_p))

# ============================================================
# 4. Substitute the particular solution into the equation
# ============================================================

yp_n = y_p
yp_n_minus1 = sp.expand_trig(y_p.subs(n, n-1))
yp_n_minus2 = sp.expand_trig(y_p.subs(n, n-2))

lhs_particular = sp.expand_trig(
    yp_n
    - sp.Rational(3,4)*yp_n_minus1
    + sp.Rational(1,8)*yp_n_minus2
)

lhs_particular = sp.expand_trig(lhs_particular).simplify()

display(Markdown("### Left-hand side"))

display(lhs_particular)

# ============================================================
# 5. Determine C3 and C4
#
# The right-hand side is:
#
# (5/8)sin(pi*n/2)
# ============================================================

rhs_particular = sp.Rational(5,8)*sp.sin(sp.pi*n/2)

# Evaluate at two convenient values of n to obtain the
# two equations for C3 and C4.

equation_C3 = sp.Eq(
    lhs_particular.subs(n, 0),
    rhs_particular.subs(n, 0)
)

equation_C4 = sp.Eq(
    lhs_particular.subs(n, 1),
    rhs_particular.subs(n, 1)
)

constants_particular = sp.solve(
    [equation_C3, equation_C4],
    [C3, C4],
    dict=True
)[0]

C3_solution = sp.factor(constants_particular[C3])
C4_solution = sp.factor(constants_particular[C4])

display(Markdown("### Constants of the particular solution"))

display(sp.Eq(C3, C3_solution))
display(sp.Eq(C4, C4_solution))

y_p = sp.simplify(
    y_p.subs(
        {
            C3: C3_solution,
            C4: C4_solution
        }
    )
)

display(sp.Eq(sp.Symbol('y_p[n]'), y_p))

# ============================================================
# 6. General solution
# ============================================================

display(Markdown("## 3. General solution"))

y_general = sp.simplify(y_h + y_p)

display(sp.Eq(sp.Symbol('y[n]'), y_general))

# ============================================================
# 7. Apply the initial conditions
#
# y[-1] = 1
# y[-2] = 1
# ============================================================

display(Markdown("## 4. Initial conditions"))

y_minus1 = sp.simplify(y_general.subs(n, -1))
y_minus2 = sp.simplify(y_general.subs(n, -2))

display(sp.Eq(sp.Symbol('y[-1]'), y_minus1))
display(sp.Eq(sp.Symbol('y[-2]'), y_minus2))

constants_initial = sp.solve(
    [
        sp.Eq(y_minus1, 1),
        sp.Eq(y_minus2, 1)
    ],
    [C1, C2],
    dict=True
)[0]

C1_solution = sp.factor(constants_initial[C1])
C2_solution = sp.factor(constants_initial[C2])

display(Markdown("### Constants of the homogeneous solution"))

display(sp.Eq(C1, C1_solution))
display(sp.Eq(C2, C2_solution))

# ============================================================
# 8. Final overall response
# ============================================================

display(Markdown("## 5. Final overall response"))

y_final = sp.simplify(
    y_general.subs(
        {
            C1: C1_solution,
            C2: C2_solution
        }
    )
)

display(sp.Eq(sp.Symbol('y[n]'), y_final))

# ============================================================
# 9. Verify the initial conditions
# ============================================================

display(Markdown("## 6. Verification of the initial conditions"))

verification_y_minus1 = sp.simplify(y_final.subs(n, -1) - 1)
verification_y_minus2 = sp.simplify(y_final.subs(n, -2) - 1)

display(sp.Eq(sp.Symbol('y[-1] - 1'), verification_y_minus1))
display(sp.Eq(sp.Symbol('y[-2] - 1'), verification_y_minus2))

# ============================================================
# 10. Verify the difference equation
#
# For n >= 0, the input is:
#
# (5/8)sin(pi*n/2)
# ============================================================

lhs_final = sp.expand_trig(
    y_final
    - sp.Rational(3,4)*y_final.subs(n, n-1)
    + sp.Rational(1,8)*y_final.subs(n, n-2)
)

rhs_final = sp.Rational(5,8)*sp.sin(sp.pi*n/2)

difference = sp.trigsimp(
    sp.expand_trig(lhs_final - rhs_final)
)

display(Markdown("## 7. Verification of the difference equation"))

display(sp.Eq(sp.Symbol('LHS'), sp.trigsimp(lhs_final)))
display(sp.Eq(sp.Symbol('RHS'), rhs_final))
display(sp.Eq(sp.Symbol('LHS - RHS'), difference))

if difference == 0:
    display(Markdown(
        "**Verification successful:** the final solution satisfies "
        "the difference equation."
    ))
else:
    display(Markdown(
        "**Verification failed.**"
    ))

# ============================================================
# 11. Transient and steady-state responses
# ============================================================

display(Markdown("## 8. Transient and steady-state responses"))

transient_response = sp.simplify(
    C1_solution*lambda1**n + C2_solution*lambda2**n
)

steady_state_response = y_p

display(
    sp.Eq(
        sp.Symbol('y_transient[n]'),
        transient_response
    )
)

display(
    sp.Eq(
        sp.Symbol('y_steady-state[n]'),
        steady_state_response
    )
)

# ============================================================
# 12. Final compact result
# ============================================================

display(Markdown("## Final result"))

display(
    sp.Eq(
        sp.Symbol('y[n]'),
        sp.factor(y_final)
    )
)
